In [ ]:
# scd type 2

# first check any update and update the new updated rows
# step 1
merge into target_table as t1
using source_table t2
on t1.id=t2.id and t1.is_active='A'

when matched and t1.name <> t2.name or t1.email<>t2.email 
or t1.last_sync_date <> t2.last_sync_date 

then update set t1.end_date=current_timestamp(),
t1.is_active='N'


## this is only for updateing the existing record , we want the new record also
# step 2
merge into target_table t1 
using source_table t2 
on t1.id=t2.id and t1.is_active='A'

when not matched then insert *

In [ ]:
## in python  Step 1 
from delta.tables import DeltaTable 

target=DeltaTable.forPath(spark,"/path/to/target")

targetdf.alias('t1').merge(sourcedf.alias("t2"),
""" t1.id=t2.id and t1.is_active='A' """) \
.whenMatchedUpdate(
    conditions=""" t1.name <> t2.name or t1.email<>t2.email or 
    t1.last_sync_date <> t2.last_sync_date""",

    set={
        "end_date":"current_timestamp()",
        "is_active":"'N'"
    }
).execute

In [ ]:
# since there is now no active ('A') row for that id, the ON condition does not match, so WHEN NOT MATCHED inserts the new active record.

In [ ]:
## step 2 
target=DeltaTable.forPath(spark,"/path/to/target")

target.alias('t1').merge(sourcedf.alias("t2"),
"""t1.id=t2.id and t1.is_active='A' """) \
.whenNotMatchedInsert(
    values={
        "id":"t2.id",
        "name":"t2.name",
         "email": "t2.email",
        "last_sync_date": "t2.last_sync_date",
        "start_date": "current_timestamp()",
        "end_date": "CAST('9999-12-31' AS TIMESTAMP)",
        "is_active": "'A'"
    }
).execute()